# KuiperAI - Real Autoregressive Language Model

This notebook trains a REAL language model with proper next-token prediction.

## What's Different:
- ✅ Real autoregressive training (predicts next token)
- ✅ Proper loss calculation (compares to actual next tokens)
- ✅ Autoregressive generation (generates one token at a time)
- ✅ Not just retrieval - actually generates new text

## Training Time:
- ~5-10 minutes on Colab
- 30 epochs
- 1,311 training examples
- Target loss: 0.5-1.0

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/Arthurc1Moude/KuiperAI-Training.git
%cd KuiperAI-Training

## Step 2: Train the Model

This will train the model with proper autoregressive objective.

Watch the loss decrease:
- Epoch 1: ~4.0 (random guessing)
- Epoch 10: ~2.0 (learning patterns)
- Epoch 20: ~1.0 (good understanding)
- Epoch 30: ~0.5-0.8 (very good)

In [ ]:
!python3 train_autoregressive.py

## Step 3: Download Trained Model

Download these files to your local machine:
- `checkpoints/autoregressive_best.pt` (trained model)
- `checkpoints/vocab.json` (vocabulary)

In [ ]:
from google.colab import files

# Download model
files.download('checkpoints/autoregressive_best.pt')
files.download('checkpoints/vocab.json')

print("✓ Files downloaded!")
print("\nNext steps:")
print("1. Put both files in your local 'checkpoints/' folder")
print("2. Run: python3 chat_autoregressive.py")
print("3. Chat with your AI!")

## Optional: Test Generation in Colab

Try generating some text to see if it works:

In [ ]:
# Quick test
import sys
sys.path.insert(0, 'src')
import numpy as np
import json
from models.transformer import Transformer

# Load vocab
with open('checkpoints/vocab.json', 'r') as f:
    vocab = json.load(f)
idx_to_token = {v: k for k, v in vocab.items()}

# Load model
model = Transformer(
    vocab_size=len(vocab),
    d_model=256,
    num_heads=8,
    num_layers=6,
    d_ff=1024,
    max_seq_length=64,
    dropout=0.3
)
model.load('checkpoints/autoregressive_best.pt')

print("✓ Model loaded!")
print(f"✓ Parameters: {model.count_parameters():,}")
print("\nTry some prompts:")

# Test prompts
test_prompts = [
    "what is machine learning",
    "explain neural networks",
    "how does ai work"
]

def tokenize(text):
    tokens = text.lower().split()
    token_ids = [vocab['<SOS>']]
    for token in tokens:
        token_ids.append(vocab.get(token, vocab['<UNK>']))
    return np.array(token_ids, dtype=np.int32)

def generate(prompt, max_tokens=30):
    tokens = tokenize(prompt).tolist()
    
    for _ in range(max_tokens):
        input_tokens = tokens[-63:]
        input_array = np.array([input_tokens], dtype=np.int32)
        
        output = model.forward(input_array)
        next_logits = output[0, -1, :] / 0.8
        
        exp_logits = np.exp(next_logits - np.max(next_logits))
        probs = exp_logits / np.sum(exp_logits)
        
        top_k_indices = np.argsort(probs)[-40:]
        top_k_probs = probs[top_k_indices]
        top_k_probs = top_k_probs / np.sum(top_k_probs)
        
        next_token = np.random.choice(top_k_indices, p=top_k_probs)
        
        if next_token == vocab['<EOS>']:
            break
        if next_token in [vocab['<PAD>'], vocab['<SOS>']]:
            continue
        
        tokens.append(next_token)
    
    generated = tokens[1:]
    text = ' '.join([idx_to_token[t] for t in generated 
                     if t not in [vocab['<PAD>'], vocab['<SOS>'], vocab['<EOS>']]])
    return text

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    response = generate(prompt)
    print(f"Response: {response}")